In [2]:
%%capture
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

In [3]:
def importar_dados(caminho):
    dados = pd.read_csv(caminho)
    dados['data_iniSE'] = pd.to_datetime(dados['data_iniSE'])
    dados = dados.set_index('data_iniSE')
    dados = dados.asfreq('W-SUN')
    dados = dados.rename(columns={'umidmed': 'umidade', 'tempmed': 'temperatura'})
    dados = dados.dropna()
    return dados

In [4]:
dados_d = importar_dados('./dados/dengue_tratado.csv')
dados_ch = importar_dados('./dados/chikungunya_tratado.csv')
dados_zk = importar_dados('./dados/zika_tratado.csv')

In [ ]:
dados_d.head()

In [6]:
def transformar_regressao(dados, lags, exog_cols=None, lags_exog=None):
    for l in range(1, lags):
        dados[f'casos_lag_{l}'] = dados['casos'].shift(l)

    if exog_cols:
        lags_exog = lags_exog or lags
        for col in exog_cols:
            for l in range(1, lags_exog):
                dados[f'{col}_lag_{l}'] = dados[col].shift(l)

    dados = dados.dropna()
    dados = dados.rename({"casos": "y"}, axis=1)
    dados = dados.drop(columns=['umidade', 'temperatura'])
    return dados

dados_d = transformar_regressao(dados_d, 6, exog_cols=['umidade', 'temperatura'])
dados_ch = transformar_regressao(dados_ch, 6, exog_cols=['umidade', 'temperatura'])
dados_zk = transformar_regressao(dados_zk, 6, exog_cols=['umidade', 'temperatura'])

In [7]:
dados_ch

,y,casos_lag_1,casos_lag_2,casos_lag_3,casos_lag_4,casos_lag_5,umidade_lag_1,umidade_lag_2,umidade_lag_3,umidade_lag_4,umidade_lag_5,temperatura_lag_1,temperatura_lag_2,temperatura_lag_3,temperatura_lag_4,temperatura_lag_5
data_iniSE,,,,,,,,,,,,,,,,
2010-02-07,0,0.0,0.0,0.0,0.0,0.0,74.697437,76.221839,80.623241,82.638528,82.989629,28.655242,28.148011,27.698323,27.503644,27.444322
2010-02-14,0,0.0,0.0,0.0,0.0,0.0,76.477691,74.697437,76.221839,80.623241,82.638528,28.252259,28.655242,28.148011,27.698323,27.503644
2010-02-21,0,0.0,0.0,0.0,0.0,0.0,76.688853,76.477691,74.697437,76.221839,80.623241,28.786932,28.252259,28.655242,28.148011,27.698323
2010-02-28,0,0.0,0.0,0.0,0.0,0.0,78.214286,76.688853,76.477691,74.697437,76.221839,28.580357,28.786932,28.252259,28.655242,28.148011
2010-03-07,0,0.0,0.0,0.0,0.0,0.0,82.148983,78.214286,76.688853,76.477691,74.697437,28.549534,28.580357,28.786932,28.252259,28.655242
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-19,0,2.0,1.0,0.0,0.0,1.0,86.020500,93.139500,91.051343,89.100671,93.843829,21.380871,21.044071,21.227457,20.741971,21.705457
2026-07-26,0,0.0,2.0,1.0,0.0,0.0,89.699886,86.020500,93.139500,91.051343,89.100671,20.946471,21.380871,21.044071,21.227457,20.741971
2026-08-02,0,0.0,0.0,2.0,1.0,0.0,89.650200,89.699886,86.020500,93.139500,91.051343,21.113829,20.946471,21.380871,21.044071,21.227457


In [8]:
def dividir_treino_teste(dados, data_limite):
    treino = dados[dados.index <= data_limite]
    teste = dados[dados.index > data_limite]
    X_train = treino.drop(["y"], axis=1)
    y_train = pd.DataFrame(treino["y"])
    X_test = teste.drop(["y"], axis=1)
    y_test = pd.DataFrame(teste["y"])
    return X_train, X_test, y_train, y_test

In [9]:
X_train_d, X_test_d, y_train_d, y_test_d = dividir_treino_teste(dados_d,'2022-01-02')

In [10]:
X_train_ch, X_test_ch, y_train_ch, y_test_ch = dividir_treino_teste(dados_ch,'2023-01-01')

In [11]:
X_train_zk, X_test_zk, y_train_zk, y_test_zk = dividir_treino_teste(dados_zk,'2023-01-01')

## Normal

In [ ]:
tscv = TimeSeriesSplit(n_splits=3)

param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [200, 500, 1000]
}

modelo_base = xgb.XGBRegressor(objective='reg:squarederror')

grid_search = GridSearchCV(
    modelo_base,
    param_grid,
    cv=tscv,
    scoring='neg_mean_squared_error'
)

grid_search.fit(X_train_zk, y_train_zk)

print(grid_search.best_params_)

In [ ]:
#Dengue: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 1000}
#Chi: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200}
#Zika: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200}

In [12]:
def treinar(X_train, y_train, X_test, ne, md, lr):
    modelo = xgb.XGBRegressor(n_estimators=ne, max_depth=md, learning_rate=lr
    ).fit(X_train, y_train, verbose=True)

    y_predito = pd.DataFrame(modelo.predict(X_test))
    y_predito.index = X_test.index
    return modelo, y_predito

In [ ]:
#Dengue: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 1000}
#Chi: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200}
#Zika: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200}

In [13]:
modelo_d, y_predito_d = treinar(X_train_d, y_train_d, X_test_d, 1000, 5, 0.01)
modelo_ch, y_predito_ch = treinar(X_train_ch, y_train_ch, X_test_ch, 200, 3, 0.01)
modelo_zk, y_predito_zk = treinar(X_train_zk, y_train_zk, X_test_zk, 200, 3, 0.01)

In [ ]:
modelo_ch.save_model('./modelos/xgboost_chik.json')

In [ ]:
def walking_forward(X_train, X_test, y_train, y_test, ne, md, lr):
    futuro = 17
    previsoes = []
    datas_previstas = []

    historico_X = X_train.copy()
    historico_y = y_train.copy()

    for i in range(0, len(X_test) - futuro + 1, futuro):
        modelo = xgb.XGBRegressor(
            n_estimators = ne,
            max_depth = md,
            learning_rate = lr,
            objective='reg:squarederror'
        )
        modelo.fit(historico_X, historico_y)

        teste_atual_X = X_test.iloc[i:i+futuro]
        teste_atual_y = y_test.iloc[i:i+futuro]

        predicao_atual = modelo.predict(teste_atual_X)

        previsoes.extend(predicao_atual)
        datas_previstas.extend(teste_atual_X.index)

        historico_X = pd.concat([historico_X, teste_atual_X])
        historico_y = pd.concat([historico_y, teste_atual_y])

    previsoes = pd.Series(previsoes, index=datas_previstas)
    reais = y_test.loc[datas_previstas]

    previsoes = pd.DataFrame(previsoes)
    reais = pd.DataFrame(reais)

    return previsoes, reais

In [ ]:
#Dengue: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 1000}
#Chi: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200}
#Zika: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200}

In [ ]:
previsoes_d, reais_d = walking_forward(X_train_d, X_test_d, y_train_d, y_test_d, 1000, 5, 0.01)
previsoes_ch, reais_ch = walking_forward(X_train_ch, X_test_ch, y_train_ch, y_test_ch, 200, 3, 0.01)
previsoes_zk, reais_zk = walking_forward(X_train_zk, X_test_zk, y_train_zk, y_test_zk, 200, 3, 0.01)

In [ ]:
def avaliar_normalizado(y_predito, y_test):
    y_predito = np.array(y_predito).ravel()
    y_test = np.array(y_test).ravel()

    mae = mean_absolute_error(y_test, y_predito)
    rmse = root_mean_squared_error(y_test, y_predito)
    r2 = r2_score(y_test, y_predito)

    des_padrao = np.std(y_test)
    nmae = mae / des_padrao
    nrmse = rmse / des_padrao


    print(f"MAE:  {nmae:.2f}")
    print(f"RMSE: {nrmse:.2f}")
    print(f"R²:   {r2:.4f}")

In [ ]:
avaliar_normalizado(previsoes_d, reais_d)

In [ ]:
avaliar_normalizado(previsoes_ch, reais_ch)

In [ ]:
avaliar_normalizado(previsoes_zk, reais_zk)

In [ ]:
def salar_dataFrames(reais, previsoes, arbo, modelo, caminho):
    dados = pd.merge(reais, previsoes, on=reais.index)
    dados = dados.rename(columns={"key_0": "data"})
    dados.index = pd.to_datetime(dados["data"])
    dados = dados.drop("data", axis=1)
    dados = dados.rename(columns={"casos": "y"})
    dados = dados.rename(columns={0: "y_pred"})
    dados.to_csv(f"{caminho}/{modelo}_{arbo}.csv", index=True)

In [ ]:
salar_dataFrames(reais_d, previsoes_d, "Dengue", "xgboost", "./resultados/xgboost")
salar_dataFrames(reais_ch, previsoes_ch, "Chikungunya", "xgboost", "./resultados/xgboost")
salar_dataFrames(reais_zk, previsoes_zk, "Zika", "xgboost", "./resultados/xgboost")

# Gráficos

In [ ]:
plt.figure(figsize=(15, 5))
plt.rcParams.update({'font.size': 16})

plt.title("Dengue com XGBoost", fontweight='bold')
plt.plot(reais_d, color="orange", label="Real")
plt.plot(previsoes_d, color="purple", marker="o", label="Predição")
plt.legend()
plt.tick_params(axis='both', labelsize=14)

plt.tight_layout()
plt.savefig('./graficos/Dengue_com_XGBoost.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
plt.rcParams.update({'font.size': 16})

plt.title("Chikungunya com XGBoost", fontweight='bold')
plt.plot(reais_ch, color="orange", label="Real")
plt.plot(previsoes_ch, color="purple", marker="o", label="Predição")
plt.legend()
plt.tick_params(axis='both', labelsize=14)

plt.tight_layout()
plt.savefig('./graficos/Chikungunya_com_XGBoost.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
plt.rcParams.update({'font.size': 16})

plt.title("Zika com XGBoost", fontweight='bold')
plt.plot(reais_zk, color="orange", label="Real")
plt.plot(previsoes_zk, color="purple", marker="o", label="Predição")
plt.legend()
plt.tick_params(axis='both', labelsize=14)

plt.tight_layout()
plt.savefig('./graficos/Zika_com_XGBoost.png', bbox_inches='tight', dpi=300)
plt.show()